## # **Lecture du bronze**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = "abfss://bronze@stbixipulse2611.dfs.core.windows.net"
TARGET = "bixi.silver.weather_hourly"

raw = spark.read.parquet(f"{BRONZE}/weather_hourly")
print(f"Bronze : {raw.count():,} lignes")

## **Le dédoublonnage, avec une window function**

In [0]:
latest_first = Window.partitionBy("observed_at").orderBy(F.col("ingested_at").desc())

silver = (
    raw.withColumn("rn", F.row_number().over(latest_first))
    .filter(F.col("rn") == 1)
    .select(
        "observed_at",
        "temperature_2m",
        "precipitation",
        "wind_speed_10m",
        "relative_humidity_2m",
        "ingested_at",
    )
)
print(f"Silver : {silver.count():,} lignes")

## **Les contrôles qualité**

In [0]:
checks = {
    "heure en double": silver.count() - silver.select("observed_at").distinct().count(),
    "température hors [-50, 50] °C": silver.filter(~F.col("temperature_2m").between(-50, 50)).count(),
    "précipitation négative": silver.filter(F.col("precipitation") < 0).count(),
    "humidité hors [0, 100] %": silver.filter(~F.col("relative_humidity_2m").between(0, 100)).count(),
}

for name, n in checks.items():
    print(f"{'✅' if n == 0 else '❌'} {name} : {n}")

failed = {name: n for name, n in checks.items() if n > 0}
assert not failed, f"Contrôles qualité en échec : {failed}"

## **L’écriture en table Delta**

In [0]:
(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)
display(spark.table(TARGET).orderBy("observed_at").limit(5))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TARGET}"))